# 04 - Register the agent with Agent 365 and send activity traces

Notebook 02 created Entra identity objects. Notebook 03a, 03b, or [03c (credential only)](03c_blueprint_credential_only.ipynb) configured a credential for the blueprint. This lesson registers **that existing child agent identity**, obtains its app-only Observability API token, and sends OpenTelemetry activity traces.

| Step | Identity used | User sign-in? |
| --- | --- | --- |
| One-time Agent 365 registration | Your authorized provisioning account and provisioning app | Yes, for this notebook's administrative setup |
| Telemetry export | Child agent identity, authenticated through its blueprint | **No**; uses a client secret or managed-identity federation |
| Optional Foundry external registration | Your Azure CLI account | Setup only |
| Optional model call | Your Azure CLI account locally, or the host managed identity on Azure | Separate from telemetry authentication |

**This notebook makes real registration writes and sends telemetry only when you run the corresponding cells.** It does not create another blueprint, change Graph/Blob permissions, deploy an endpoint, create an AI Teammate, or publish a hosted Foundry agent. Agent 365 receives activity **spans**, not every `print()` or Python application log.

Start with a **fresh kernel**, not the kernel used by notebook 03. OpenTelemetry initialization must happen before importing Agent Framework or the model SDK.

## 0. Prerequisites

Complete notebook 02 and [03c - Blueprint credential only](03c_blueprint_credential_only.ipynb) for the telemetry-only path. Alternatively, reuse the credential setup from 03a (local client secret) or 03b (Azure managed identity). You do not need their Graph OBO/user-login or Blob sections for telemetry.

Your tenant must have Agent 365 enabled, the required licensing/preview access, and administrator acceptance of its terms. An authorized administrator must add and consent Microsoft Graph **delegated** `AgentRegistration.ReadWrite.All` and `User.ReadBasic.All` on the normal provisioning app from notebook 02 (`ENTRA_PROVISIONING_CLIENT_ID`). Keep its existing `AgentIdentity.Read.All` permission. `User.ReadBasic.All` reads the existing sponsor's display name and UPN; it is used only during setup. Its desktop redirect URI remains `http://localhost:8400`. **Do not put these administrative permissions on the runtime child agent.** The notebook 02 bootstrap script does not add these two permissions.

Registration uses `POST /beta/copilot/agentRegistrations`, the current Agent 365 registration contract. This is an **undocumented preview Graph API**, verified against Microsoft's [registration script](https://github.com/microsoft/Agent365-devTools/blob/main/scripts/bulk-agent-registration/New-A365AgentRegistration.ps1), not a stable public Graph contract. Tenant eligibility, permissions, and the contract can change. A 403/404 is a setup/access failure, not proof that no registration exists. We do not silently fall back to the legacy API or create another identity.

The installed CLI may be older than the `a365 setup all --agent-registration-only` workflow. This notebook uses REST with explicit IDs instead; it does not upgrade the CLI or run full provisioning.

From the project directory in PowerShell, install the updated dependencies and then restart the `.venv` kernel:

```powershell
uv pip install --python .venv\Scripts\python.exe --index-url https://packagefeedproxy.microsoft.io/pypi/simple -r requirements.txt
```

This lesson targets the **commercial cloud**. Government cloud tenants require their own authority, Graph, and Observability resource configuration.

## 1. Configuration

Reuse the IDs in `.env` from notebook 02 and add the telemetry options below. **Do not re-enter the sponsor's name, email, or ID.** Section 2 reads the sponsor already assigned to the child agent in Entra and caches the profile in the ignored `.agent365-registration.json`. Subsequent runs reuse that cache without Graph access or a sponsor sign-in. These details are telemetry attribution, not a user token.

```dotenv
ENTRA_TENANT_ID=YOUR-TENANT-GUID
ENTRA_BLUEPRINT_CLIENT_ID=YOUR-BLUEPRINT-APPLICATION-GUID
ENTRA_AGENT_IDENTITY_OBJECT_ID=YOUR-CHILD-AGENT-GUID
ENTRA_PROVISIONING_CLIENT_ID=YOUR-NOTEBOOK-02-PROVISIONING-APP-GUID
AGENT_IDENTITY_NAME=LearningAssistantLocal
A365_AUTH_MODE=client_secret
A365_CAPTURE_CONTENT=false
A365_EXPORT_TO_FOUNDRY=true
APPLICATIONINSIGHTS_CONNECTION_STRING='YOUR-FOUNDRY-LINKED-APPLICATION-INSIGHTS-CONNECTION-STRING'
```

This walkthrough exports to **both Agent 365 and Foundry / Application Insights** by default. Copy the connection string from the Application Insights resource linked to your Foundry project into your local `.env`, replacing the placeholder; do not paste it into notebook code or outputs. If `.env` already contains `A365_EXPORT_TO_FOUNDRY=false` from an earlier lesson, change it to `true`. Existing environment variables also take precedence, so update any conflicting value and restart the kernel. Set the flag to `false` only if you deliberately want Agent 365 alone.

Only if the child has **multiple sponsors**, set optional `A365_SPONSOR_USER_ID` to select one of those existing users. This does not assign or change sponsors. Old `A365_SPONSOR_USER_NAME` and `A365_SPONSOR_USER_EMAIL` settings are no longer used. Rerun REGISTER to refresh the cache after a sponsor change.

For **client-secret setup in 03a or 03c**, retain `ENTRA_BLUEPRINT_CLIENT_SECRET` in the ignored `.env`; never paste its value into a cell. For **managed-identity setup in 03b or 03c**, set `A365_AUTH_MODE=managed_identity` and retain `AZURE_MANAGED_IDENTITY_CLIENT_ID`. Run on the Azure host with that identity attached and its federated trust configured on the blueprint. There is no credential fallback or interactive telemetry login.

The child identity object ID is also its client ID. It is **not** the blueprint client ID. `A365_CAPTURE_CONTENT=false` redacts manually recorded prompts, tool results, and responses. Setting it to `true` exports that content; use only non-sensitive examples and consider tenant data residency.

Run from the project directory. `.env` does not override existing environment variables. After changing configuration, restart the kernel. Keep `.env` and the ignored `.agent365-registration.json` state out of shared folders and screenshots.

In [ ]:
import asyncio
import base64
import json
import logging
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from threading import Lock
from urllib.parse import quote, urlsplit
from uuid import UUID, uuid4

import msal
import requests
from azure.core.credentials import AccessToken
from azure.identity import ManagedIdentityCredential
from dotenv import load_dotenv

if any(name == 'agent_framework' or name.startswith('agent_framework.') for name in sys.modules):
    raise RuntimeError('Restart the kernel; initialize tracing before importing Agent Framework.')

load_dotenv('.env')

def required(name):
    value = os.getenv(name, '').strip()
    if not value:
        raise ValueError(f'Set {name} in .env, then restart the kernel.')
    return value

tenant_id = str(UUID(required('ENTRA_TENANT_ID')))
blueprint_client_id = str(UUID(required('ENTRA_BLUEPRINT_CLIENT_ID')))
agent_identity_id = str(UUID(required('ENTRA_AGENT_IDENTITY_OBJECT_ID')))
agent_name = os.getenv('AGENT_IDENTITY_NAME', 'LearningAssistantLocal').strip()
auth_mode = required('A365_AUTH_MODE').lower()
if auth_mode not in {'client_secret', 'managed_identity'}:
    raise ValueError('A365_AUTH_MODE must be client_secret or managed_identity.')
if agent_identity_id == blueprint_client_id or not agent_name:
    raise ValueError('Use a named child agent identity, not the blueprint ID.')
if auth_mode == 'client_secret':
    required('ENTRA_BLUEPRINT_CLIENT_SECRET')
else:
    str(UUID(required('AZURE_MANAGED_IDENTITY_CLIENT_ID')))

capture_content = os.getenv('A365_CAPTURE_CONTENT', 'false').lower() == 'true'
export_to_foundry = os.getenv('A365_EXPORT_TO_FOUNDRY', 'true').lower() == 'true'
if export_to_foundry:
    required('APPLICATIONINSIGHTS_CONNECTION_STRING')
state_path = Path('.agent365-registration.json')
graph_base = 'https://graph.microsoft.com'
registration_path = '/beta/copilot/agentRegistrations'
observability_app_id = '9b975845-388f-4429-889e-eab1ef63949c'
observability_scope = f'api://{observability_app_id}/.default'

print('Telemetry authentication:', auth_mode)
print('Exporting child identity:', agent_identity_id)
print('Foundry trace export:', export_to_foundry)
print('Prompt/response content capture:', capture_content)

## 2. Register the existing agent

The next cell defines the administrative helper but does not sign in or write. The confirmation cell checks the child's blueprint relationship and expands its **existing sponsors**, then reads the selected sponsor's basic user profile. It does not substitute the currently signed-in administrator or the blueprint's owner for the child's sponsor, and does not change Entra sponsorship. Missing/incomplete sponsor information, insufficient read permissions, and ambiguous multiple sponsors stop setup rather than inventing attribution.

Setup caches the sponsor alongside the tenant, child, blueprint, and a client-generated registration ID **before** POST, registers, and reads back the object. Reruns reuse this state and verify the identity before accepting success. An HTTP 409 is accepted only when the service returns an existing ID that can be read back. The sponsor read uses the [agent identity GET operation's supported `$expand`](https://learn.microsoft.com/graph/api/agentidentity-get?view=graph-rest-1.0); if your tenant does not authorize that expansion, stop and resolve setup access with your administrator rather than escalating runtime permissions.

`managedByAppId` is the **calling provisioning app's client ID**, following Microsoft's standalone registration script. Do not impersonate the CLI's first-party managed-by app ID. `sourceAgentId` and `agentIdentityId` are the child identity; `agentIdentityBlueprintId` is the blueprint client ID.

An ambiguous network failure leaves the saved request ID intact. Rerun to probe it; do not delete the state to force a new registration. Only a confirmed 404 permits another POST with the same request ID. Read failures and mismatched registrations stop the notebook.

Type **REGISTER** for first-time setup or to refresh the cached sponsor, even if the agent is already registered elsewhere. Setup uses **MSAL's browser authorization-code flow**, with PKCE and state validation, instead of `InteractiveBrowserCredential`. It keeps the same provisioning app, Graph scopes, and `http://localhost:8400` desktop redirect. No device-code flow or runtime user authentication is introduced.

The cell prints `http://localhost:8400?welcome=true` before opening the browser. If no tab appears, open that local URL manually **on the computer running this kernel** and select **Continue to Microsoft sign-in**. Complete sign-in/consent and leave the cell running until the callback completes. The link only works during that sign-in attempt; do not visit the bare callback URL to test it. Browser authentication times out after **120 seconds**; authority discovery and Graph requests have separate bounded network timeouts. Interrupt an earlier attempt before rerunning, because both use port 8400. Do not share the sign-in page or its address bar.

Progress messages distinguish authority discovery, browser launch, and Graph operations. If sign-in fails, the cell reports the OAuth/AADSTS code without exposing tokens or directory IDs. Resolve consent/Conditional Access errors with your administrator; the notebook does not grant itself permissions.

Press Enter on later runs to reuse the local sponsor cache **without signing in or calling Graph**. The first run of this updated notebook also needs REGISTER to upgrade older state files that lack sponsor details. Skipping does not claim registration verification; telemetry authorization still depends on the actual service registration.

In [ ]:
registration_scopes = (
    f'{graph_base}/AgentRegistration.ReadWrite.All',
    f'{graph_base}/AgentIdentity.Read.All',
    f'{graph_base}/User.ReadBasic.All',
)

def graph_request(graph_token, method, path, body=None):
    resource = 'agent registration'
    if '/servicePrincipals/' in path:
        resource = 'child identity and sponsors'
    elif '/users/' in path:
        resource = 'sponsor profile'
    print(f'Graph {method}: {resource}...', flush=True)
    try:
        response = requests.request(
            method, graph_base + path,
            headers={'Authorization': f'Bearer {graph_token}', 'Content-Type': 'application/json'},
            json=body, timeout=30, allow_redirects=False,
        )
    except requests.RequestException:
        raise RuntimeError('Graph request failed; keep registration state and rerun. Request details are suppressed.') from None
    print(f'Graph {method}: {resource} returned HTTP {response.status_code}.', flush=True)
    try:
        result = response.json() if response.content else {}
    except ValueError:
        result = {}
    if not isinstance(result, dict):
        raise RuntimeError('Graph returned an unexpected response shape.')
    if not 200 <= response.status_code < 300 and response.status_code not in {404, 409}:
        code = result.get('error', {}).get('code', 'unknown')
        raise RuntimeError(f'Graph failed: HTTP {response.status_code}, code={code}. Check provisioning-app consent, caller role, and tenant eligibility.')
    return response.status_code, result

def save_registration_state(state):
    temporary = state_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, indent=2), encoding='utf-8')
    temporary.replace(state_path)

def load_registration_state():
    state = json.loads(state_path.read_text(encoding='utf-8')) if state_path.exists() else {
        'tenantId': tenant_id,
        'agentIdentityId': agent_identity_id,
        'blueprintClientId': blueprint_client_id,
        'requestId': str(uuid4()),
    }
    expected = {'tenantId': tenant_id, 'agentIdentityId': agent_identity_id, 'blueprintClientId': blueprint_client_id}
    if any(state.get(key) != value for key, value in expected.items()):
        raise RuntimeError('Registration state belongs to another tenant/agent. Use a separate working directory.')
    return state

def resolve_existing_sponsor(graph_token, child):
    if 'sponsors' not in child or child.get('sponsors@odata.nextLink'):
        raise RuntimeError('Could not read the complete existing sponsor relationship. Check setup read permissions; no sponsor inferred.')
    candidates = {
        str(UUID(item['id'])) for item in child['sponsors']
        if item.get('@odata.type') in {None, '#microsoft.graph.directoryObject', '#microsoft.graph.user'}
    }
    selected_id = os.getenv('A365_SPONSOR_USER_ID', '').strip()
    if selected_id:
        selected_id = str(UUID(selected_id))
        if selected_id not in candidates:
            raise RuntimeError('The selected user is not an existing sponsor of this child agent. No sponsorship changed.')
    elif len(candidates) == 1:
        selected_id = next(iter(candidates))
    else:
        raise RuntimeError('Expected one existing human sponsor. If there are multiple, select one with A365_SPONSOR_USER_ID; if none, check provisioning.')
    status, profile = graph_request(
        graph_token, 'GET', f'/v1.0/users/{selected_id}?$select=id,displayName,userPrincipalName,mail',
    )
    if status != 200 or str(profile.get('id', '')).lower() != selected_id:
        raise RuntimeError('Could not read the selected sponsor as a user. Check setup User.ReadBasic.All permission.')
    name = profile.get('displayName')
    email = profile.get('userPrincipalName') or profile.get('mail')
    if not name or not email:
        raise RuntimeError('Sponsor profile is incomplete. Check setup user-profile read access; no name or UPN inferred.')
    return {'id': selected_id, 'displayName': name, 'userPrincipalName': email}

def verify_registration(record):
    expected = {
        'sourceAgentId': agent_identity_id,
        'agentIdentityId': agent_identity_id,
        'agentIdentityBlueprintId': blueprint_client_id,
    }
    if any(str(record.get(key, '')).lower() != value.lower() for key, value in expected.items()):
        raise RuntimeError('Registration readback does not match the configured child and blueprint. No success claimed.')

def register_existing_agent(graph_token, provisioning_client_id):
    state = load_registration_state()
    status, child = graph_request(
        graph_token, 'GET',
        f'/v1.0/servicePrincipals/{agent_identity_id}/microsoft.graph.agentIdentity?$select=id,agentIdentityBlueprintId&$expand=sponsors($select=id)',
    )
    if status != 200 or str(child.get('agentIdentityBlueprintId', '')).lower() != blueprint_client_id.lower():
        raise RuntimeError('Could not verify that this child identity belongs to this blueprint.')

    state['sponsor'] = resolve_existing_sponsor(graph_token, child)
    request_id = str(UUID(state['requestId']))
    registration_id = state.get('registrationId', request_id)
    status, record = graph_request(graph_token, 'GET', f'{registration_path}/{quote(registration_id, safe="")}')
    if status == 404 and not registration_id.startswith('T_'):
        status, record = graph_request(graph_token, 'GET', f'{registration_path}/T_{quote(registration_id, safe="")}')
    if status != 200:
        if status != 404:
            raise RuntimeError('Registration probe was inconclusive; no POST attempted.')
        save_registration_state(state)
        timestamp = datetime.now(timezone.utc).isoformat()
        payload = {
            'id': request_id,
            'displayName': agent_name,
            'description': 'Agent Framework learning agent; app-only Agent 365 telemetry.',
            'ownerIds': [state['sponsor']['id']],
            'createdBy': state['sponsor']['id'],
            'sourceCreatedDateTime': timestamp,
            'sourceLastModifiedDateTime': timestamp,
            'sourceAgentId': agent_identity_id,
            'agentIdentityId': agent_identity_id,
            'agentIdentityBlueprintId': blueprint_client_id,
            'managedByAppId': provisioning_client_id,
        }
        status, record = graph_request(graph_token, 'POST', registration_path, payload)
        if status not in {200, 201, 409} or not record.get('id'):
            raise RuntimeError(f'Registration is not verified (HTTP {status}, no usable ID). Keep state; check admin center before retrying.')
        registration_id = str(record['id'])
        state['registrationId'] = registration_id
        save_registration_state(state)
        status, record = graph_request(graph_token, 'GET', f'{registration_path}/{quote(registration_id, safe="")}')
    if status != 200:
        raise RuntimeError('Registration write returned, but readback failed. Keep state and retry after propagation.')
    verify_registration(record)
    state['registrationId'] = str(record.get('id') or registration_id)
    save_registration_state(state)
    return state['registrationId']


In [ ]:
print('Registration target:', tenant_id, agent_name, agent_identity_id)
if input('Type REGISTER to register/read back this agent, or Enter to skip: ').strip() == 'REGISTER':
    provisioning_client_id = str(UUID(required('ENTRA_PROVISIONING_CLIENT_ID')))
    print('REGISTER received. Discovering the provisioning-account authority...', flush=True)
    registration_app = None
    sign_in_result = {}
    try:
        registration_app = msal.PublicClientApplication(
            provisioning_client_id, authority=f'https://login.microsoftonline.com/{tenant_id}',
            timeout=(10, 30),
        )
        print('Opening browser sign-in. If no tab appears, open http://localhost:8400?welcome=true on this computer.', flush=True)
        sign_in_result = registration_app.acquire_token_interactive(
            scopes=list(registration_scopes), port=8400, timeout=120, prompt='select_account',
            welcome_template='<html><body><h2>Agent 365 setup</h2><a href="$auth_uri">Continue to Microsoft sign-in</a><p>Keep this page and its address private.</p></body></html>',
            auth_uri_callback=lambda _: print('Automatic browser launch failed. Open http://localhost:8400?welcome=true manually.', flush=True),
        )
        if not sign_in_result.get('access_token'):
            code = sign_in_result.get('error', 'unknown')
            aadsts = sign_in_result.get('error_codes', [])
            raise RuntimeError(f'Provisioning sign-in failed: {code}, AADSTS={aadsts}. Check provisioning-app consent, redirect URI, and Conditional Access.')
        print('Provisioning sign-in complete. Checking the agent, sponsor, and registration...', flush=True)
        registration_id = register_existing_agent(sign_in_result['access_token'], provisioning_client_id)
    except msal.BrowserInteractionTimeoutError:
        raise RuntimeError('Browser sign-in timed out. Rerun this cell and open its local welcome link; finish sign-in within 120 seconds.') from None
    except requests.RequestException:
        raise RuntimeError('Could not reach the provisioning sign-in service. Check Entra connectivity, proxy, and TLS configuration, then retry.') from None
    except OSError:
        raise RuntimeError('Could not start browser sign-in or its local callback. Interrupt any earlier sign-in using port 8400, then retry.') from None
    finally:
        sign_in_result.clear()
        registration_app = None
    print('Agent 365 registration verified:', registration_id)
else:
    print('Registration skipped; no administrative sign-in or Graph request performed. Existing service registration is still required.')

sponsor = load_registration_state().get('sponsor', {})
if not sponsor.get('id') or not sponsor.get('displayName') or not sponsor.get('userPrincipalName'):
    raise RuntimeError('No cached sponsor profile. Run this cell with REGISTER once to read the sponsor already assigned in Entra.')
sponsor_id = str(UUID(sponsor['id']))
sponsor_name = sponsor['displayName']
sponsor_email = sponsor['userPrincipalName']
print('Using the cached existing sponsor for telemetry attribution; no sponsor token is needed.')

## 3. Acquire the Observability API token without a user

This uses the same autonomous two-request pattern as the app-only branch of notebook 03:

1. Blueprint credential + `fmi_path=CHILD-AGENT-ID` -> exchange token T1 (`api://AzureADTokenExchange/.default`).
2. Child client ID + T1 as `client_assertion` -> **Observability API token** (`api://9b975845-388f-4429-889e-eab1ef63949c/.default`).

The managed-identity variant first gets the host's managed-identity assertion, then performs the same two requests. Do not reuse a Storage, Graph, blueprint, or delegated OBO token as the final telemetry token.

The synchronous resolver caches in memory, refreshes before expiry, and checks the `(agent, tenant)` tuple. It can run on the exporter's worker thread; it never opens a browser. The lock prevents concurrent refreshes. Authentication failures are not converted into fake success or another identity.

JWT payload decoding below is only a **diagnostic sanity check**, not signature validation or an authorization decision. The service validates the token. No token or credential is printed or written to disk. Registered blueprint-derived instances normally need no separate `Agent365.Observability.OtelWrite` grant on the S2S route, subject to service policy.

In [ ]:
token_lock = Lock()
cached_observability_token = None
managed_identity = None

def token_request(form):
    token_url = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    try:
        response = requests.post(token_url, data=form, timeout=30, allow_redirects=False)
    except requests.RequestException:
        raise RuntimeError('Entra token request failed; details suppressed to protect credentials.') from None
    try:
        result = response.json()
    except ValueError:
        raise RuntimeError(f'Entra returned a non-JSON response (HTTP {response.status_code}).') from None
    if response.status_code != 200 or not result.get('access_token'):
        raise RuntimeError(
            f'Entra token acquisition failed (HTTP {response.status_code}); '
            f'code={result.get("error", "unknown")}; AADSTS={result.get("error_codes", [])}. '
            'Check blueprint credentials, federation, identity IDs, and Conditional Access.'
        )
    if result.get('token_type', '').lower() != 'bearer' or int(result.get('expires_in', 0)) <= 60:
        raise RuntimeError('Entra did not return a usable bearer token lifetime.')
    return result

def acquire_observability_token():
    global managed_identity
    parent_form = {
        'grant_type': 'client_credentials',
        'client_id': blueprint_client_id,
        'scope': 'api://AzureADTokenExchange/.default',
        'fmi_path': agent_identity_id,
    }
    if auth_mode == 'client_secret':
        parent_form['client_secret'] = required('ENTRA_BLUEPRINT_CLIENT_SECRET')
    else:
        if managed_identity is None:
            managed_identity = ManagedIdentityCredential(client_id=required('AZURE_MANAGED_IDENTITY_CLIENT_ID'))
        parent_form['client_assertion_type'] = 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer'
        parent_form['client_assertion'] = managed_identity.get_token('api://AzureADTokenExchange/.default').token
    parent = token_request(parent_form)
    requested_at = time.time()
    result = token_request({
        'grant_type': 'client_credentials',
        'client_id': agent_identity_id,
        'client_assertion_type': 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer',
        'client_assertion': parent['access_token'],
        'scope': observability_scope,
    })
    try:
        encoded = result['access_token'].split('.')[1]
        claims = json.loads(base64.urlsafe_b64decode(encoded + '=' * (-len(encoded) % 4)))
    except (ValueError, IndexError, UnicodeDecodeError):
        raise RuntimeError('Expected a JWT Observability access token; no token displayed.') from None
    if (
        claims.get('aud') not in {observability_app_id, f'api://{observability_app_id}'}
        or str(claims.get('azp') or claims.get('appid', '')).lower() != agent_identity_id.lower()
        or str(claims.get('tid', '')).lower() != tenant_id.lower()
        or 'scp' in claims
    ):
        raise RuntimeError('Token is not an app-only Observability token for the configured child and tenant.')
    expires_on = min(requested_at + int(result['expires_in']), int(claims['exp']))
    if expires_on <= time.time() + 60:
        raise RuntimeError('Observability token is already expired or too close to expiry.')
    return AccessToken(result['access_token'], int(expires_on))

def resolve_observability_token(requested_agent_id, requested_tenant_id):
    global cached_observability_token
    if (requested_agent_id, requested_tenant_id) != (agent_identity_id, tenant_id):
        return None
    with token_lock:
        if cached_observability_token is None or cached_observability_token.expires_on <= time.time() + 60:
            cached_observability_token = acquire_observability_token()
        return cached_observability_token.token


In [ ]:
await asyncio.to_thread(resolve_observability_token, agent_identity_id, tenant_id)
print('Acquired app-only Observability token for the child agent. Token value is not displayed.')
print('Expires UTC:', datetime.fromtimestamp(cached_observability_token.expires_on, timezone.utc).isoformat())
print('Token acquisition does not prove registration authorization; the export is the next check.')

## 4. Configure the exporter once

Both activation flags and `a365_use_s2s_endpoint=True` are explicit. Manual `InvokeAgentScope`, `InferenceScope`, and `ExecuteToolScope` make the lesson's tracing visible in code. Distro auto-instrumentation is disabled here. Section 6 also explicitly disables Agent Framework's own default instrumentation so it cannot duplicate the manual spans or record content independently. Manual content recording respects `A365_CAPTURE_CONTENT`.

The Azure Monitor exporter is **enabled by default** and sends the **same spans** to the Application Insights resource linked to your Foundry project, alongside Agent 365 export. Configure `APPLICATIONINSIGHTS_CONNECTION_STRING` in section 1 before running. Section 7's Foundry External Agent registration is optional and separate from export. If the destination resource disables local authentication, Azure Monitor also requires an authorized credential; a connection string alone is not sufficient.

Offline retry storage is disabled in this learning notebook so failed payloads and message content are not persisted. Enable durable delivery deliberately in production, with protected storage and an appropriate retry policy. Console logging stays local. This kernel explicitly disables OTel log/metric export and SDK usage-metric collection so the lesson exports **only traces**, including when Azure Monitor is enabled.

In [ ]:
if globals().get('tracing_initialized', False):
    raise RuntimeError('Tracing was already initialized. Restart the kernel to reconfigure it.')

os.environ.setdefault('OTEL_LOG_LEVEL', 'INFO')
os.environ.setdefault('A365_OBSERVABILITY_LOG_LEVEL', 'info|warn|error')
os.environ['OTEL_LOGS_EXPORTER'] = 'none'
os.environ['OTEL_METRICS_EXPORTER'] = 'none'
os.environ['APPLICATIONINSIGHTS_STATSBEAT_DISABLED_ALL'] = 'true'
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('microsoft.opentelemetry.a365.core.exporters.agent365_exporter').setLevel(logging.DEBUG)
from microsoft.opentelemetry import use_microsoft_opentelemetry
from microsoft.opentelemetry.a365.core import (
    AgentDetails, BaggageBuilder, CallerDetails, Channel, ExecuteToolScope,
    InferenceCallDetails, InferenceOperationType, InferenceScope,
    InvokeAgentScope, InvokeAgentScopeDetails, Request, ToolCallDetails, UserDetails,
)
from opentelemetry import trace

use_microsoft_opentelemetry(
    enable_a365=True,
    a365_enable_observability_exporter=True,
    a365_use_s2s_endpoint=True,
    a365_token_resolver=resolve_observability_token,
    a365_exporter_disable_offline_storage=True,
    enable_azure_monitor=export_to_foundry,
    azure_monitor_connection_string=os.getenv('APPLICATIONINSIGHTS_CONNECTION_STRING') if export_to_foundry else None,
    azure_monitor_exporter_disable_offline_storage=True,
    azure_monitor_enable_live_metrics=False,
    azure_monitor_enable_performance_counters=False,
    disable_logging=True,
    disable_metrics=True,
    sampling_ratio=1.0,
    enable_sensitive_data=False,
    instrumentation_options={
        name: {'enabled': False} for name in (
            'agent_framework', 'openai', 'openai_agents', 'langchain', 'semantic_kernel',
            'requests', 'httpx', 'httpx2', 'urllib', 'urllib3', 'fastapi', 'flask', 'django', 'psycopg2',
        )
    },
)
tracing_initialized = True
agent_details = AgentDetails(
    agent_id=agent_identity_id, agent_name=agent_name,
    agent_description='Agent Framework learning agent',
    agent_blueprint_id=blueprint_client_id, tenant_id=tenant_id,
)
user_details = UserDetails(user_id=sponsor_id, user_name=sponsor_name, user_email=sponsor_email)
caller_details = CallerDetails(user_details=user_details)

def telemetry_content(text):
    return text if capture_content else '[content redacted]'

def turn_baggage(conversation_id):
    return (
        BaggageBuilder().tenant_id(tenant_id).agent_id(agent_identity_id)
        .agent_blueprint_id(blueprint_client_id).agent_name(agent_name)
        .conversation_id(conversation_id).session_id(conversation_id)
        .user_id(sponsor_id).user_name(sponsor_name).user_email(sponsor_email).build()
    )

def lookup_learning_topic():
    return 'Agent 365 registers agent identities and receives authenticated activity traces.'

async def flush_traces():
    drained = await asyncio.to_thread(trace.get_tracer_provider().force_flush, timeout_millis=120_000)
    if not drained:
        raise RuntimeError('Trace queue did not drain within the timeout. Check exporter diagnostics.')
    print('Trace queue drained. This alone does NOT prove HTTP acceptance or portal indexing; check exporter diagnostics.')

print('Agent 365 S2S tracing configured; caller attribution uses the sponsor, not a delegated token.')

## 5. Send a smoke-test activity (no model, no user sign-in)

Running this cell **sends a real test activity** to Agent 365. The baggage context wraps the invocation, tool, and output. The child ID is used everywhere; the blueprint remains a separate roll-up attribute.

This is a local deterministic operation, not a model response. We do **not** fabricate an inference span or token counts. The tool is called explicitly by the notebook, not selected by an LLM. Section 6 adds a genuine inference.

In [ ]:
smoke_conversation_id = str(uuid4())
smoke_request = Request(
    content=telemetry_content('Check Agent 365 telemetry configuration.'),
    session_id=smoke_conversation_id, conversation_id=smoke_conversation_id,
    channel=Channel(name='jupyter'),
)
with turn_baggage(smoke_conversation_id):
    with InvokeAgentScope.start(smoke_request, InvokeAgentScopeDetails(), agent_details, caller_details) as invocation:
        invocation.record_attributes({'learning.smoke_test': True})
        with ExecuteToolScope.start(
            smoke_request, ToolCallDetails(tool_name='lookup_learning_topic', tool_type='function'),
            agent_details, user_details,
        ) as tool_scope:
            topic = lookup_learning_topic()
            tool_scope.record_response(telemetry_content(topic))
        invocation.record_output_messages([telemetry_content(topic)])
print(topic)
print('Smoke-test conversation:', smoke_conversation_id)
await flush_traces()

## 6. Optional: trace a real Agent Framework / Foundry model call

The smoke test already exercises authenticated Agent 365 delivery. Run the next cell only if you also want a real `chat` inference span. It calls the model configured by notebook 01 and can incur model charges.

Set `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_MODEL` as before. Locally, the model uses your existing `az login` session. On the managed-identity path, it uses the host MI, which needs its **own Foundry model-access role**. Neither registration nor the Observability token grants model access. The model caller is intentionally separate from the child identity used for telemetry.

This tutor agent has no framework-managed tools and makes one model call. The local lookup is performed explicitly first, and its result is supplied as context. Manual inference wrapping is appropriate for this single-call example; for an agent with multiple LLM/tool calls, instrument each call or switch to framework auto-instrumentation instead of wrapping the entire loop as one inference.

Type **RUN** to call the model; Enter skips. The agent ID is pinned to the child ID, not a new random framework identity.

In [ ]:
if input('Type RUN to make one model call, or Enter to skip: ').strip() == 'RUN':
    from agent_framework import Agent
    from agent_framework.foundry import FoundryChatClient
    from agent_framework.observability import disable_instrumentation
    from azure.identity.aio import AzureCliCredential, ManagedIdentityCredential as AsyncManagedIdentityCredential

    disable_instrumentation()
    project_endpoint = required('FOUNDRY_PROJECT_ENDPOINT').rstrip('/')
    parsed_endpoint = urlsplit(project_endpoint)
    if parsed_endpoint.scheme != 'https' or not parsed_endpoint.hostname or not parsed_endpoint.path.startswith('/api/projects/') or parsed_endpoint.query or parsed_endpoint.fragment:
        raise ValueError('Use the HTTPS Foundry project endpoint from notebook 01.')
    deployment_name = required('FOUNDRY_MODEL')
    model_credential = (
        AsyncManagedIdentityCredential(client_id=required('AZURE_MANAGED_IDENTITY_CLIENT_ID'))
        if auth_mode == 'managed_identity' else AzureCliCredential(tenant_id=tenant_id)
    )
    client = FoundryChatClient(project_endpoint=project_endpoint, model=deployment_name, credential=model_credential)
    try:
        agent = Agent(
            client=client, id=agent_identity_id, name=agent_name,
            instructions='You are a patient technical tutor. Explain the provided topic briefly and accurately.',
        )
        conversation_id = str(uuid4())
        prompt = 'Explain how my agent can send activity traces without asking a user to sign in.'
        request = Request(
            content=telemetry_content(prompt), session_id=conversation_id,
            conversation_id=conversation_id, channel=Channel(name='jupyter'),
        )
        with turn_baggage(conversation_id):
            with InvokeAgentScope.start(request, InvokeAgentScopeDetails(), agent_details, caller_details) as invocation:
                with ExecuteToolScope.start(
                    request, ToolCallDetails(tool_name='lookup_learning_topic', tool_type='function'),
                    agent_details, user_details,
                ) as tool_scope:
                    topic = lookup_learning_topic()
                    tool_scope.record_response(telemetry_content(topic))
                with InferenceScope.start(
                    request, InferenceCallDetails(
                        operationName=InferenceOperationType.CHAT,
                        model=deployment_name, providerName='azure.ai.foundry',
                    ), agent_details, user_details,
                ) as inference:
                    inference.record_input_messages([telemetry_content(f'{prompt}\nContext: {topic}')])
                    response = await agent.run(f'{prompt}\nContext: {topic}')
                    inference.record_output_messages([telemetry_content(response.text)])
                    usage = response.usage_details or {}
                    if usage.get('input_token_count') is not None:
                        inference.record_input_tokens(usage['input_token_count'])
                    if usage.get('output_token_count') is not None:
                        inference.record_output_tokens(usage['output_token_count'])
                invocation.record_output_messages([telemetry_content(response.text)])
        print(response.text)
    finally:
        await client.close()
        await model_credential.close()
        await flush_traces()
else:
    print('Model call skipped. The smoke test does not require Foundry credentials.')

## 7. Optional: link the same telemetry identity as a Foundry External Agent

**Yes, these can be used together, but the registrations and destinations are separate.**

| Integration | Purpose | Requirement |
| --- | --- | --- |
| Agent 365 registration + S2S exporter | Enterprise inventory and agent activity | Registered Entra child identity and app-only Observability token |
| Foundry External Agent (preview) | Foundry trace view and trace-based evaluation for an agent running outside Foundry | External registration plus spans in the project's linked Application Insights |

Use **the same child GUID** as Foundry's `otel_agent_id` and every span's `gen_ai.agent.id`. Do not replace the Agent 365 identity with the Foundry display name. Setting `otel_agent_id` correlates traces; it does not assign an Entra identity, change credentials, deploy your code, or grant Agent 365 export permission.

Before running this section, enable the optional Azure Monitor export in section 4 and restart the kernel. Your Foundry project must have that Application Insights resource connected. Set `FOUNDRY_PROJECT_ENDPOINT` and optionally `FOUNDRY_EXTERNAL_AGENT_NAME=LearningAssistantExternal`. The setup account needs **Foundry User or higher** on the project and an Azure CLI sign-in.

This preview uses `AIProjectClient(..., allow_preview=True)` and `ExternalAgentDefinition(otel_agent_id=agent_identity_id)`. The cell reads first; it reuses an exact existing external registration and refuses to replace a different kind or identity. Only a confirmed 404 permits creation. There is no automatic publish or evaluation job.

Microsoft's [Foundry/Agent 365 integration table](https://learn.microsoft.com/azure/foundry/agents/concepts/agent-365-integration) documents registry sync for published **prompt and hosted agents**; it does not promise that linking an external agent alone registers this child in Agent 365. Keep section 2 explicit.

See [Register external agents for observability and evaluation](https://learn.microsoft.com/azure/foundry/agents/how-to/register-external-agent). This external registration is an observability/evaluation record, **not a remotely callable endpoint or an A2A connection**.

In [ ]:
if input('Type LINK to create/verify the Foundry external registration, or Enter to skip: ').strip() == 'LINK':
    if not export_to_foundry:
        raise RuntimeError('Enable A365_EXPORT_TO_FOUNDRY and configure Application Insights, then restart and rerun.')
    import re
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import ExternalAgentDefinition
    from azure.core.exceptions import ResourceNotFoundError
    from azure.identity import AzureCliCredential as SyncAzureCliCredential

    external_name = os.getenv('FOUNDRY_EXTERNAL_AGENT_NAME', 'LearningAssistantExternal')
    if not re.fullmatch(r'[A-Za-z0-9_-]+', external_name):
        raise ValueError('External agent name must use letters, numbers, underscores, or hyphens.')
    with SyncAzureCliCredential(tenant_id=tenant_id) as setup_credential:
        with AIProjectClient(
            endpoint=required('FOUNDRY_PROJECT_ENDPOINT'), credential=setup_credential, allow_preview=True,
        ) as project:
            try:
                existing = project.agents.get(agent_name=external_name)
            except ResourceNotFoundError:
                project.agents.create_version(
                    agent_name=external_name,
                    description='Externally running learning agent; telemetry identity shared with Agent 365.',
                    definition=ExternalAgentDefinition(otel_agent_id=agent_identity_id),
                )
                existing = project.agents.get(agent_name=external_name)
            definition = existing.versions.latest.definition
            if not isinstance(definition, ExternalAgentDefinition) or definition.otel_agent_id != agent_identity_id:
                raise RuntimeError('Existing Foundry agent has a different kind or telemetry ID. Nothing overwritten.')
    print('Foundry External Agent verified:', external_name)
    print('Matching gen_ai.agent.id:', agent_identity_id)
else:
    print('Foundry external registration skipped; Agent 365 export remains independent.')

## 8. Verify the result

**Separate four checks:** registration readback, token acquisition, HTTP export acceptance, and portal indexing. Success at one does not prove the next. `force_flush()` only proves the processors completed their queued work, not that the backend accepted it. This notebook enables DEBUG on the specific exporter logger because SDK 1.6 logs successful sends at that level. Look for `HTTP 200 success` (or another 2xx success), and check for HTTP errors or rejected spans; no success line means delivery has not been verified.

- In Agent 365/Microsoft 365 admin center, check the registered agent and its Activity view after ingestion. Security hunting availability depends on tenant licensing/configuration. Filter by **the child `agent_identity_id`**, not the blueprint GUID. Indexing is asynchronous and can take tens of minutes; HTTP acceptance does not guarantee immediate visibility.
- In Foundry, open **Build > Agents > your external agent > Traces** after Application Insights ingestion, normally a few minutes. The project must link the exact destination resource; `otel_agent_id` must match the child's `gen_ai.agent.id`. Registration without Azure Monitor export will not show these traces.
- For Foundry trace-based evaluation, inputs/outputs and conversation IDs must satisfy its schema. The default redaction here limits content-based evaluation; only enable capture deliberately for non-sensitive data.

| Symptom | Check |
| --- | --- |
| Registration 403/404 | Consent on the **provisioning app**, caller role/eligibility, Agent 365 enablement, and preview API access; no fallback provisioning |
| Registration write succeeds but readback fails | Keep saved state; retry after propagation, without claiming verified registration |
| Token acquisition fails | Correct child and blueprint IDs, secret/trust, managed-identity host, and Conditional Access |
| Export 401 | App-only Observability audience, token expiry, child identity, and explicit S2S route |
| Export 403 `insufficient_scope` | Verify Agent 365 registration/propagation and tenant service policy; token acquisition alone is insufficient |
| Spans skipped / no identity groups | Baggage must wrap scope creation; tenant and child IDs must be present |
| HTTP export accepted, Activity empty | Correct child filter, caller/sponsor attributes, invocation/output spans, tenant enablement, and ingestion delay |

For persistent authorization failures, follow the documented administrator-approved application-role fallback; do not switch telemetry to delegated OBO or request a user login. See [direct OTel authentication recipes](https://learn.microsoft.com/microsoft-agent-365/developer/direct-open-telemetry-integration#authentication-recipes) and [observability authentication setup](https://learn.microsoft.com/microsoft-agent-365/developer/observability-authentication-setup).

This notebook has not verified live delivery merely by being created. Run it against your eligible tenant and inspect the export diagnostics before treating the integration as complete.

## 9. Finish

Flush and close the telemetry providers when finished. This does **not** delete the Agent 365 or optional Foundry registrations. Restart the kernel before sending more traces after shutdown. Clear all notebook outputs before sharing; IDs, sponsor attribution, model output, and error diagnostics can contain tenant information.

In [ ]:
await flush_traces()
await asyncio.to_thread(trace.get_tracer_provider().shutdown)
with token_lock:
    cached_observability_token = None
    if managed_identity is not None:
        managed_identity.close()
        managed_identity = None
print('Telemetry shut down. Registrations retained; restart the kernel before another run.')